# 🎵 BeatmapBERT — RunPod Training Notebook
### NVIDIA RTX PRO 4500 Blackwell | 20 Epoch, No Validation
---
**Data sudah di-download dan di-preprocess sebelumnya.**

**Flow**: Install → Write Source → Write Config → Train → Evaluate


## 1. Install Dependencies
> ⚠️ Setelah cell ini selesai, **RESTART KERNEL** (Kernel → Restart Kernel), lalu lanjut ke Cell 2.


In [ ]:
# Step 1: Upgrade typing_extensions + install PyTorch untuk Blackwell (sm_120)
!pip install --upgrade typing_extensions
!pip uninstall -y torch torchvision torchaudio
!pip install --pre torch torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
!pip install -q numpy>=1.26 scipy>=1.11 pandas>=2.2 librosa>=0.10 soundfile>=0.12 mido>=1.3 PyYAML>=6.0 tqdm>=4.66 scikit-learn>=1.4 matplotlib>=3.8 rich>=13.7

print("\n✅ Install selesai!")
print("⚠️  SEKARANG RESTART KERNEL (Kernel → Restart Kernel)")
print("⚠️  Lalu jalankan Cell 2 (skip cell ini)")


## 2. Verify GPU (jalankan SETELAH restart kernel)


In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA capability: sm_{props.major}{props.minor}")
print("\n✅ Ready!")


## 3. Setup Project Directory


In [ ]:
import os
PROJECT_ROOT = '/workspace/beatmap_bert_project_real'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
for d in ['src/beatbert', 'src/beatbert/configs', 'src/beatbert/data', 'src/beatbert/models',
          'src/beatbert/training', 'src/beatbert/utils', 'src/beatbert/inference', 'configs', 'scripts',
          'checkpoints', 'data/processed', 'data/splits', 'data/predictions']:
    os.makedirs(d, exist_ok=True)
print(f"✓ Working dir: {os.getcwd()}")


## 4. Write Project Source Code


In [ ]:
%%writefile src/beatbert/__init__.py
__version__ = '0.1.0'



In [ ]:
%%writefile src/beatbert/configs/__init__.py
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict

import yaml


def load_config(path: str | Path) -> Dict[str, Any]:
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


def resolve_paths(config: Dict[str, Any], project_root: str | Path) -> Dict[str, Any]:
    root = Path(project_root)
    cfg = dict(config)
    cfg['project_root'] = str(root)
    cfg['paths'] = dict(config['paths'])
    for key, value in cfg['paths'].items():
        cfg['paths'][key] = str((root / value).resolve())
        Path(cfg['paths'][key]).mkdir(parents=True, exist_ok=True)
    return cfg



In [ ]:
%%writefile src/beatbert/data/__init__.py



In [ ]:
%%writefile src/beatbert/data/indexer.py
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

AUDIO_EXTS = {'.wav', '.mp3', '.flac', '.ogg'}
MIDI_EXTS = {'.mid', '.midi'}


def scan_raw_pairs(raw_dir: str | Path) -> List[Dict[str, str]]:
    raw_path = Path(raw_dir)
    files = list(raw_path.rglob('*'))
    audio_map = {}
    midi_map = {}
    for p in files:
        if not p.is_file():
            continue
        if p.suffix.lower() in AUDIO_EXTS:
            audio_map[p.stem] = p
        elif p.suffix.lower() in MIDI_EXTS:
            midi_map[p.stem] = p
    shared = sorted(set(audio_map) & set(midi_map))
    pairs = []
    for stem in shared:
        pairs.append({'song_id': stem, 'audio_path': str(audio_map[stem]), 'midi_path': str(midi_map[stem])})
    return pairs



In [ ]:
%%writefile src/beatbert/data/preprocess.py
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import gc
import pandas as pd
from tqdm import tqdm

from beatbert.data.indexer import scan_raw_pairs
from beatbert.utils.audio import extract_audio_features, frame_times_ms
from beatbert.utils.io import save_npz
from beatbert.utils.midi import notes_to_frame_labels, parse_midi_notes


def preprocess_dataset(cfg: Dict) -> pd.DataFrame:
    """Preprocessing sederhana: WAV + MIDI → NPZ (tanpa augmentasi)."""
    raw_dir = cfg['paths']['raw_dir']
    processed_dir = Path(cfg['paths']['processed_dir'])
    processed_dir.mkdir(parents=True, exist_ok=True)

    pairs = scan_raw_pairs(raw_dir)
    if not pairs:
        raise FileNotFoundError(f'No matching WAV/MIDI pairs found in {raw_dir}')

    rows: List[Dict] = []
    for item in tqdm(pairs, desc='Preprocessing songs'):
        song_id = item['song_id']
        try:
            notes = parse_midi_notes(item['midi_path'], cfg)
            features = extract_audio_features(item['audio_path'], cfg)
            f_ms = frame_times_ms(features.mel.shape[1], cfg)
            labels = notes_to_frame_labels(notes, f_ms, cfg)

            out_path = processed_dir / f'{song_id}.npz'
            save_npz(
                out_path,
                mel=features.mel.astype('float32'),
                frame_times_ms=f_ms.astype('float32'),
                event=labels['event'].astype('float32'),
                lane=labels['lane'].astype('int64'),
                onset_times_ms=features.onset_times_ms.astype('float32'),
                beat_times_ms=features.beat_times_ms.astype('float32'),
                bpm=[features.bpm],
                duration_ms=[features.duration_ms],
                audio_path=[item['audio_path']],
                midi_path=[item['midi_path']],
                song_id=[song_id],
            )
            rows.append({
                'song_id': song_id,
                'audio_path': item['audio_path'],
                'midi_path': item['midi_path'],
                'processed_path': str(out_path),
                'num_frames': int(features.mel.shape[1]),
                'num_notes': int(labels['event'].sum()),
                'duration_ms': float(features.duration_ms),
                'bpm': float(features.bpm),
            })
        except Exception as e:
            print(f'[SKIP] {song_id}: {e}')
        finally:
            # Bersihkan RAM setelah setiap lagu
            gc.collect()

    df = pd.DataFrame(rows).sort_values('song_id').reset_index(drop=True)
    df.to_csv(processed_dir / 'metadata.csv', index=False)
    print(f'Berhasil preprocessing {len(rows)}/{len(pairs)} lagu.')
    return df



In [ ]:
%%writefile src/beatbert/data/splits.py
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from sklearn.model_selection import train_test_split

from beatbert.data.indexer import scan_raw_pairs


def make_splits(cfg: Dict) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    pairs = scan_raw_pairs(cfg['paths']['raw_dir'])
    if len(pairs) < 3:
        raise ValueError('Need at least 3 songs to create train/val/test splits.')
    df = pd.DataFrame(pairs)
    ratios = cfg['dataset']['split_ratio']
    train_df, temp_df = train_test_split(df, test_size=(1 - ratios['train']), random_state=cfg['seed'], shuffle=True)
    val_share = ratios['val'] / (ratios['val'] + ratios['test'])
    val_df, test_df = train_test_split(temp_df, test_size=(1 - val_share), random_state=cfg['seed'], shuffle=True)

    split_dir = Path(cfg['paths']['splits_dir'])
    split_dir.mkdir(parents=True, exist_ok=True)
    for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        split_df[['song_id']].to_csv(split_dir / f'{name}.csv', index=False)
    return train_df, val_df, test_df


def load_split_song_ids(split_dir: str | Path, split_name: str) -> List[str]:
    path = Path(split_dir) / f'{split_name}.csv'
    if not path.exists():
        raise FileNotFoundError(f'Split file not found: {path}')
    df = pd.read_csv(path)
    return df['song_id'].astype(str).tolist()



In [ ]:
%%writefile src/beatbert/data/dataset.py
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset

from beatbert.data.splits import load_split_song_ids
from beatbert.utils.io import load_npz


class BeatmapDataset(Dataset):
    def __init__(self, cfg: Dict, split_name: str):
        self.cfg = cfg
        self.split_name = split_name

        processed_dir = Path(cfg['paths']['processed_dir'])
        song_ids = load_split_song_ids(cfg['paths']['splits_dir'], split_name)
        self.song_paths = [p for p in (processed_dir / f'{sid}.npz' for sid in song_ids) if p.exists()]
        self.context_frames = int(cfg['dataset']['context_frames'])
        stride = int(cfg['dataset']['train_stride'] if split_name == 'train' else cfg['dataset']['eval_stride'])

        # Negative sample ratio: berapa persen sample tanpa event yang dipertahankan
        # Untuk training, kita tetap simpan beberapa sample negatif agar model
        # belajar bahwa TIDAK SEMUA frame adalah event.
        # Tapi terlalu banyak negative sample = model terlalu bias ke 0.
        neg_ratio = float(cfg['dataset'].get('negative_sample_ratio', 0.15))

        self.samples: List[Tuple[Path, int, int]] = []
        positive_samples: List[Tuple[Path, int, int]] = []
        negative_samples: List[Tuple[Path, int, int]] = []

        for path in self.song_paths:
            try:
                with np.load(path) as data:
                    total_frames = data['mel'].shape[1]
                    event_all = data['event']

                    candidate_samples: List[Tuple[Path, int, int]] = []

                    if total_frames <= self.context_frames:
                        candidate_samples.append((path, 0, total_frames))
                    else:
                        for start in range(0, total_frames - self.context_frames + 1, stride):
                            candidate_samples.append((path, start, start + self.context_frames))

                        if candidate_samples and candidate_samples[-1][2] < total_frames:
                            candidate_samples.append((path, total_frames - self.context_frames, total_frames))

                    for sample_path, start, end in candidate_samples:
                        event_slice = event_all[start:end]
                        if np.sum(event_slice) > 0:
                            positive_samples.append((sample_path, start, end))
                        else:
                            negative_samples.append((sample_path, start, end))
            except Exception as e:
                print(f'[SKIP corrupt] {path.name}: {e}')

        # Tambahkan semua positive samples
        self.samples.extend(positive_samples)

        # Tambahkan sebagian negative samples (untuk keseimbangan)
        # Ini penting agar model belajar bahwa tidak semua frame adalah event
        if negative_samples and neg_ratio > 0:
            rng = np.random.RandomState(cfg.get('seed', 42))
            n_neg = max(1, int(len(positive_samples) * neg_ratio))
            n_neg = min(n_neg, len(negative_samples))
            neg_indices = rng.choice(len(negative_samples), size=n_neg, replace=False)
            for idx in neg_indices:
                self.samples.append(negative_samples[idx])

        # Shuffle samples
        rng = np.random.RandomState(cfg.get('seed', 42))
        rng.shuffle(self.samples)

        print(
            f'[{split_name}] positive samples: {len(positive_samples)}, '
            f'negative samples kept: {len(self.samples) - len(positive_samples)}/{len(negative_samples)}, '
            f'total: {len(self.samples)}'
        )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        path, start, end = self.samples[idx]

        # Lazy loading: buka file NPZ, ambil hanya slice yang dibutuhkan,
        # lalu segera tutup file handle agar RAM tidak membengkak.
        with np.load(path) as data:
            mel = data['mel'][:, start:end]
            event = data['event'][start:end]
            lane = data['lane'][start:end]
            frame_times = data['frame_times_ms'][start:end]

        valid_len = mel.shape[1]

        if valid_len < self.context_frames:
            pad = self.context_frames - valid_len
            mel = np.pad(mel, ((0, 0), (0, pad)))
            event = np.pad(event, (0, pad))
            lane = np.pad(lane, (0, pad), constant_values=-100)
            frame_times = np.pad(frame_times, (0, pad), constant_values=-1)
            attention_mask = np.concatenate([
                np.ones(valid_len, dtype=np.float32),
                np.zeros(pad, dtype=np.float32)
            ])
        else:
            attention_mask = np.ones(self.context_frames, dtype=np.float32)

        return {
            'mel': torch.from_numpy(mel).float(),
            'event': torch.from_numpy(event).float(),
            'lane': torch.from_numpy(lane).long(),
            'attention_mask': torch.from_numpy(attention_mask).float(),
            'frame_times_ms': torch.from_numpy(frame_times).float(),
        }


In [ ]:
%%writefile src/beatbert/models/__init__.py



In [ ]:
%%writefile src/beatbert/models/cnn_frontend.py
from __future__ import annotations

import torch
from torch import nn


class CNNFrontend(nn.Module):
    def __init__(self, input_mels: int, channels: list[int], d_model: int, dropout: float = 0.1):
        super().__init__()
        c1, c2, c3 = channels
        self.net = nn.Sequential(
            nn.Conv2d(1, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1),
            nn.GELU(),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Conv2d(c2, c3, kernel_size=3, padding=1),
            nn.BatchNorm2d(c3),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Dropout(dropout),
        )
        reduced_mels = input_mels // 4
        self.proj = nn.Linear(c3 * reduced_mels, d_model)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        # mel: [B, M, T]
        x = mel.unsqueeze(1)
        x = self.net(x)  # [B, C, M', T]
        x = x.permute(0, 3, 1, 2).contiguous()  # [B, T, C, M']
        x = x.flatten(start_dim=2)  # [B, T, C*M']
        return self.proj(x)



In [ ]:
%%writefile src/beatbert/models/transformer.py
from __future__ import annotations

import torch
from torch import nn


class PositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        return x + self.embedding(positions)


class BeatTransformerEncoder(nn.Module):
    def __init__(self, d_model: int, num_layers: int, num_heads: int, ff_mult: int, dropout: float):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * ff_mult,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        padding_mask = None
        if attention_mask is not None:
            padding_mask = attention_mask == 0
        return self.encoder(x, src_key_padding_mask=padding_mask)



In [ ]:
%%writefile src/beatbert/models/beatmap_model.py
from __future__ import annotations

from typing import Dict

import torch
from torch import nn

from beatbert.models.cnn_frontend import CNNFrontend
from beatbert.models.transformer import BeatTransformerEncoder, PositionalEmbedding


class BeatmapModel(nn.Module):
    def __init__(self, cfg: Dict):
        super().__init__()
        mcfg = cfg['model']
        self.frontend = CNNFrontend(
            input_mels=mcfg['input_mels'],
            channels=mcfg['cnn_channels'],
            d_model=mcfg['d_model'],
            dropout=mcfg['dropout'],
        )
        self.positional = PositionalEmbedding(mcfg['max_seq_len'], mcfg['d_model'])
        self.encoder = BeatTransformerEncoder(
            d_model=mcfg['d_model'],
            num_layers=mcfg['num_layers'],
            num_heads=mcfg['num_heads'],
            ff_mult=mcfg['ff_mult'],
            dropout=mcfg['dropout'],
        )
        self.norm = nn.LayerNorm(mcfg['d_model'])
        self.event_head = nn.Linear(mcfg['d_model'], 1)
        self.lane_head = nn.Linear(mcfg['d_model'], mcfg['num_lanes'])
        self.predict_note_type = bool(mcfg.get('predict_note_type', True))
        self.type_head = nn.Linear(mcfg['d_model'], 2) if self.predict_note_type else None

    def forward(self, mel: torch.Tensor, attention_mask: torch.Tensor | None = None) -> Dict[str, torch.Tensor]:
        x = self.frontend(mel)
        x = self.positional(x)
        x = self.encoder(x, attention_mask=attention_mask)
        x = self.norm(x)
        out = {
            'event_logits': self.event_head(x).squeeze(-1),
            'lane_logits': self.lane_head(x),
        }
        if self.predict_note_type:
            out['type_logits'] = self.type_head(x)
        return out



In [ ]:
%%writefile src/beatbert/training/__init__.py



In [ ]:
%%writefile src/beatbert/training/losses.py
from __future__ import annotations
from typing import Dict
import torch
import torch.nn.functional as F


def sigmoid_focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.75,
    gamma: float = 2.0,
    reduction: str = 'mean',
) -> torch.Tensor:
    """Sigmoid Focal Loss — lebih efektif dari BCE untuk class imbalance extreme.

    Focal Loss mengurangi bobot pada easy negatives yang mendominasi dataset,
    supaya model fokus belajar dari hard examples (frame di sekitar note boundary).

    Args:
        logits: raw logits sebelum sigmoid, shape [B, T]
        targets: ground truth 0/1, shape [B, T]
        alpha: weight untuk positive class (0.75 = positif 3x lebih penting)
        gamma: focusing parameter (2.0 = standar, semakin besar = semakin fokus ke hard examples)
    """
    probs = torch.sigmoid(logits)
    ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

    # p_t = probability of correct class
    p_t = probs * targets + (1 - probs) * (1 - targets)

    # Focal modulating factor: (1 - p_t)^gamma
    # Easy examples (p_t tinggi) → weight kecil
    # Hard examples (p_t rendah) → weight besar
    focal_weight = (1 - p_t) ** gamma

    # Alpha weighting: positive class gets alpha, negative gets (1-alpha)
    alpha_weight = alpha * targets + (1 - alpha) * (1 - targets)

    focal_loss = alpha_weight * focal_weight * ce_loss

    if reduction == 'mean':
        return focal_loss.mean()
    elif reduction == 'sum':
        return focal_loss.sum()
    return focal_loss


def compute_losses(
    outputs: Dict[str, torch.Tensor],
    batch: Dict[str, torch.Tensor],
    cfg: Dict,
    pos_weight: torch.Tensor
) -> Dict[str, torch.Tensor]:
    event_target = batch['event'].float().clamp(0, 1)
    lane_target = batch['lane']

    event_logits = outputs['event_logits']
    lane_logits = outputs['lane_logits']

    # =========================
    # EVENT LOSS
    # =========================
    use_focal = cfg['train'].get('use_focal_loss', True)

    if use_focal:
        focal_alpha = float(cfg['train'].get('focal_alpha', 0.75))
        focal_gamma = float(cfg['train'].get('focal_gamma', 2.0))
        event_loss = sigmoid_focal_loss(
            event_logits,
            event_target,
            alpha=focal_alpha,
            gamma=focal_gamma,
        )
    else:
        # Fallback ke BCE dengan pos_weight
        pw = torch.nan_to_num(pos_weight, nan=1.0, posinf=100.0, neginf=1.0)
        event_loss = F.binary_cross_entropy_with_logits(
            event_logits,
            event_target,
            pos_weight=pw,
        )

    # =========================
    # LANE LOSS
    # =========================
    label_smoothing = float(cfg['train'].get('label_smoothing', 0.0))

    if (lane_target != -100).any():
        lane_loss = F.cross_entropy(
            lane_logits.transpose(1, 2),
            lane_target,
            ignore_index=-100,
            label_smoothing=label_smoothing,
        )
    else:
        lane_loss = torch.tensor(0.0, device=lane_logits.device)

    # =========================
    # TOTAL LOSS
    # =========================
    total = (
        cfg['train']['event_loss_weight'] * event_loss
        + cfg['train']['lane_loss_weight'] * lane_loss
    )

    return {
        'loss': total,
        'event_loss': event_loss,
        'lane_loss': lane_loss,
    }


In [ ]:
%%writefile src/beatbert/training/metrics.py
from __future__ import annotations

from typing import Dict, List, Tuple

import numpy as np
import torch


def frame_classification_metrics(event_logits: torch.Tensor, lane_logits: torch.Tensor, event_target: torch.Tensor, lane_target: torch.Tensor, threshold: float = 0.5) -> Dict[str, float]:
    event_prob = torch.sigmoid(event_logits)
    event_pred = (event_prob >= threshold).long()
    event_true = event_target.long()

    tp = int(((event_pred == 1) & (event_true == 1)).sum().item())
    fp = int(((event_pred == 1) & (event_true == 0)).sum().item())
    fn = int(((event_pred == 0) & (event_true == 1)).sum().item())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-8, precision + recall)

    lane_pred = lane_logits.argmax(dim=-1)
    mask = lane_target != -100
    lane_acc = float((lane_pred[mask] == lane_target[mask]).float().mean().item()) if mask.any() else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'lane_acc': lane_acc}


def decode_events(event_logits: np.ndarray, lane_logits: np.ndarray, frame_times_ms: np.ndarray, threshold: float) -> List[Tuple[float, int]]:
    event_prob = 1.0 / (1.0 + np.exp(-event_logits))
    idxs = np.where((event_prob >= threshold) & (frame_times_ms >= 0))[0]
    lanes = lane_logits.argmax(axis=-1)
    return [(float(frame_times_ms[i]), int(lanes[i])) for i in idxs]


def event_level_metrics(pred_events: List[Tuple[float, int]], true_events: List[Tuple[float, int]], tolerance_ms: float) -> Dict[str, float]:
    matched_true = set()
    tp = 0
    lane_tp = 0
    offsets = []
    for pt, plane in pred_events:
        best_j = None
        best_dt = None
        for j, (tt, tlane) in enumerate(true_events):
            if j in matched_true:
                continue
            dt = abs(pt - tt)
            if dt <= tolerance_ms and (best_dt is None or dt < best_dt):
                best_dt = dt
                best_j = j
        if best_j is not None:
            matched_true.add(best_j)
            tp += 1
            offsets.append(best_dt)
            if pred_events and plane == true_events[best_j][1]:
                lane_tp += 1
    fp = len(pred_events) - tp
    fn = len(true_events) - tp
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-8, precision + recall)
    median_offset = float(np.median(offsets)) if offsets else float('nan')
    mae_offset = float(np.mean(offsets)) if offsets else float('nan')
    lane_acc = lane_tp / max(1, tp)
    return {
        'event_precision': precision,
        'event_recall': recall,
        'event_f1': f1,
        'lane_event_acc': lane_acc,
        'median_offset_ms': median_offset,
        'mae_offset_ms': mae_offset,
    }



In [ ]:
%%writefile src/beatbert/training/trainer.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict

import torch
from rich.console import Console
from torch.utils.data import DataLoader
from tqdm import tqdm

from beatbert.data.dataset import BeatmapDataset
from beatbert.models.beatmap_model import BeatmapModel
from beatbert.training.losses import compute_losses
from beatbert.training.metrics import frame_classification_metrics

console = Console()


def _auto_device(cfg: Dict) -> torch.device:
    requested = cfg['general']['device']
    if requested != 'auto':
        return torch.device(requested)
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _estimate_pos_weight(train_dataset: BeatmapDataset) -> float:
    """Estimasi pos_weight dengan sampling cepat (tidak perlu baca semua file)."""
    import numpy as np
    pos = 0.0
    neg = 0.0
    # Sampling maksimal 200 file, bukan seluruh dataset
    all_paths = list({s[0] for s in train_dataset.samples})  # unique paths
    rng = np.random.RandomState(42)
    sample_paths = rng.choice(all_paths, size=min(200, len(all_paths)), replace=False)
    for path in sample_paths:
        with np.load(path) as data:
            event = data['event']
            pos += float((event == 1).sum())
            neg += float((event == 0).sum())
    return max(1.0, neg / max(1.0, pos))


def _build_scheduler(optimizer: torch.optim.Optimizer, cfg: Dict, steps_per_epoch: int):
    """Cosine Annealing LR scheduler with linear warmup.

    Warmup membantu training stabil di awal epoch pertama.
    Cosine annealing menurunkan LR secara gradual untuk konvergensi yang lebih baik.
    """
    warmup_epochs = int(cfg['train'].get('warmup_epochs', 3))
    total_epochs = int(cfg['train']['epochs'])
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = total_epochs * steps_per_epoch

    def lr_lambda(current_step: int) -> float:
        if current_step < warmup_steps:
            # Linear warmup: dari 0 ke 1
            return float(current_step) / max(1, warmup_steps)
        # Cosine annealing: dari 1 ke min_lr_ratio
        progress = float(current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        min_lr_ratio = float(cfg['train'].get('min_lr_ratio', 0.01))
        import math
        return min_lr_ratio + 0.5 * (1.0 - min_lr_ratio) * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@dataclass
class TrainArtifacts:
    model: BeatmapModel
    device: torch.device


def build_dataloaders(cfg: Dict):
    train_ds = BeatmapDataset(cfg, 'train')
    val_ds = BeatmapDataset(cfg, 'val')
    nw = int(cfg['general']['num_workers'])
    loader_kwargs = dict(
        batch_size=cfg['train']['batch_size'],
        num_workers=nw,
        pin_memory=cfg['general']['pin_memory'],
        persistent_workers=(nw > 0),   # Keep workers alive — hindari overhead respawn
        prefetch_factor=4 if nw > 0 else None,  # Pre-load 4 batch ke depan
    )
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    
    # Val_loader juga pakai worker agar tidak bottleneck di disk I/O.
    # Tanpa worker, setiap NPZ file dibaca di main thread → 15-20 menit gap antar epoch!
    val_nw = max(2, nw // 2)  # Minimal 2 worker, atau setengah dari train workers
    val_loader = DataLoader(
        val_ds,
        shuffle=False,
        batch_size=cfg['train']['batch_size'],
        num_workers=val_nw,
        pin_memory=cfg['general']['pin_memory'],
        persistent_workers=(val_nw > 0),
        prefetch_factor=4 if val_nw > 0 else None,
    )
    return train_ds, val_ds, train_loader, val_loader


def train(cfg: Dict) -> TrainArtifacts:
    device = _auto_device(cfg)
    console.print(f'[bold green]Device:[/bold green] {device}')

    train_ds, val_ds, train_loader, val_loader = build_dataloaders(cfg)
    console.print(f'[cyan]Train samples: {len(train_ds)}, Val samples: {len(val_ds)}[/cyan]')

    model = BeatmapModel(cfg).to(device)

    # Log jumlah parameter
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    console.print(f'[cyan]Model params: {trainable_params:,} trainable / {total_params:,} total[/cyan]')

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg['train']['lr'],
        weight_decay=cfg['train']['weight_decay'],
    )

    # LR Scheduler
    scheduler = _build_scheduler(optimizer, cfg, steps_per_epoch=len(train_loader))

    scaler = torch.amp.GradScaler('cuda', enabled=bool(cfg['train']['amp']) and device.type == 'cuda')

    if cfg['train']['event_positive_weight'] == 'auto':
        pos_weight_value = _estimate_pos_weight(train_ds)
        console.print(f'[cyan]Auto pos_weight: {pos_weight_value:.2f}[/cyan]')
    else:
        pos_weight_value = float(cfg['train']['event_positive_weight'])
    pos_weight = torch.tensor(pos_weight_value, device=device)

    use_focal = cfg['train'].get('use_focal_loss', True)
    if use_focal:
        console.print(
            f'[bold magenta]Using Focal Loss[/bold magenta] '
            f'alpha={cfg["train"].get("focal_alpha", 0.75)}, '
            f'gamma={cfg["train"].get("focal_gamma", 2.0)}'
        )

    best_val_f1 = 0.0
    patience = 0
    ckpt_dir = Path(cfg['paths']['checkpoint_dir'])
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    monitor = cfg['train'].get('monitor_metric', 'f1')  # 'f1' or 'loss'
    
    start_epoch = 1
    resume_path = ckpt_dir / 'last.pt'
    if resume_path.exists():
        console.print(f'[bold yellow]Auto-Resume aktif: Melanjutkan dari {resume_path}[/bold yellow]')
        state = torch.load(resume_path, map_location=device)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        scheduler.load_state_dict(state['scheduler_state_dict'])
        start_epoch = state['epoch'] + 1
        if monitor == 'f1' and 'val_metrics' in state:
            best_val_f1 = state['val_metrics']['f1']
        elif monitor == 'loss' and 'val_loss' in state:
            best_val_f1 = state['val_loss']

    for epoch in range(start_epoch, cfg['train']['epochs'] + 1):
        # ============ TRAINING ============
        model.train()
        train_loss = 0.0
        train_event_loss = 0.0
        train_lane_loss = 0.0

        pbar = tqdm(train_loader, desc=f'Train {epoch}/{cfg["train"]["epochs"]}')
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=scaler.is_enabled()):
                outputs = model(batch['mel'], batch['attention_mask'])
                losses = compute_losses(outputs, batch, cfg, pos_weight)

            scaler.scale(losses['loss']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['train']['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += float(losses['loss'].item())
            train_event_loss += float(losses['event_loss'].item())
            train_lane_loss += float(losses['lane_loss'].item())

            current_lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix(loss=f'{losses["loss"].item():.4f}', lr=f'{current_lr:.6f}')

        n_batches = max(1, len(train_loader))
        avg_train = train_loss / n_batches
        avg_event = train_event_loss / n_batches
        avg_lane = train_lane_loss / n_batches

        # ============ VALIDATION ============
        validate_every = int(cfg['train'].get('validate_every', 1))  # 0 = skip, N = setiap N epoch
        run_val = validate_every > 0 and (epoch % validate_every == 0 or epoch == cfg['train']['epochs'])

        if run_val:
            val_loss, val_metrics = validate(model, val_loader, device, cfg, pos_weight)

            current_lr = optimizer.param_groups[0]['lr']
            console.print(
                f'[bold cyan]Epoch {epoch:3d}[/bold cyan] '
                f'lr={current_lr:.6f} '
                f'train_loss={avg_train:.4f} (event={avg_event:.4f} lane={avg_lane:.4f}) '
                f'val_loss={val_loss:.4f} '
                f'val_f1={val_metrics["f1"]:.4f} '
                f'precision={val_metrics["precision"]:.4f} '
                f'recall={val_metrics["recall"]:.4f} '
                f'lane_acc={val_metrics["lane_acc"]:.4f}'
            )
        else:
            val_loss = 0.0
            val_metrics = {'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'lane_acc': 0.0}
            current_lr = optimizer.param_groups[0]['lr']
            console.print(
                f'[bold cyan]Epoch {epoch:3d}[/bold cyan] '
                f'lr={current_lr:.6f} '
                f'train_loss={avg_train:.4f} (event={avg_event:.4f} lane={avg_lane:.4f}) '
                f'[dim](validation skipped)[/dim]'
            )

        # ============ CHECKPOINTING ============
        state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': cfg,
            'val_metrics': val_metrics,
            'val_loss': val_loss,
        }
        torch.save(state, ckpt_dir / 'last.pt')

        if not run_val:
            # Tanpa validasi, simpan best.pt di epoch terakhir saja
            if epoch == cfg['train']['epochs']:
                torch.save(state, ckpt_dir / 'best.pt')
                console.print(f'  [green]✓ Final model saved as best.pt[/green]')
            continue

        # Monitor berdasarkan F1 (lebih tepat untuk imbalanced data)
        if monitor == 'f1':
            improved = val_metrics['f1'] > best_val_f1
            if improved:
                best_val_f1 = val_metrics['f1']
        else:
            improved = val_loss < best_val_f1 if best_val_f1 > 0 else True
            if epoch == 1:
                best_val_f1 = val_loss

        if improved:
            patience = 0
            torch.save(state, ckpt_dir / 'best.pt')
            console.print(f'  [green]✓ Best model saved (f1={val_metrics["f1"]:.4f})[/green]')
        else:
            patience += 1
            console.print(f'  [yellow]No improvement ({patience}/{cfg["train"]["early_stopping_patience"]})[/yellow]')
            if patience >= cfg['train']['early_stopping_patience']:
                console.print('[bold yellow]Early stopping triggered.[/bold yellow]')
                break


    console.print(f'[bold green]Training complete. Best val F1: {best_val_f1:.4f}[/bold green]')
    return TrainArtifacts(model=model, device=device)


def validate(model: BeatmapModel, loader: DataLoader, device: torch.device, cfg: Dict, pos_weight: torch.Tensor):
    model.eval()
    losses_accum = []
    metrics_accum = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(batch['mel'], batch['attention_mask'])
            losses = compute_losses(outputs, batch, cfg, pos_weight)
            metrics = frame_classification_metrics(
                outputs['event_logits'].detach().cpu(),
                outputs['lane_logits'].detach().cpu(),
                batch['event'].detach().cpu(),
                batch['lane'].detach().cpu(),
                threshold=cfg['eval']['event_threshold'],
            )
            losses_accum.append(float(losses['loss'].item()))
            metrics_accum.append(metrics)
    mean_loss = sum(losses_accum) / max(1, len(losses_accum))
    mean_metrics = {
        k: sum(m[k] for m in metrics_accum) / max(1, len(metrics_accum))
        for k in metrics_accum[0]
    }
    return mean_loss, mean_metrics



In [ ]:
%%writefile src/beatbert/utils/__init__.py



In [ ]:
%%writefile src/beatbert/utils/audio.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import librosa
import numpy as np


@dataclass
class AudioFeatures:
    waveform: np.ndarray
    sr: int
    duration_ms: float
    mel: np.ndarray
    onset_times_ms: np.ndarray
    beat_times_ms: np.ndarray
    bpm: float


def load_audio(audio_path: str | Path, cfg: Dict) -> Tuple[np.ndarray, int]:
    audio_cfg = cfg['audio']
    y, sr = librosa.load(
        str(audio_path),
        sr=audio_cfg['sample_rate'],
        mono=audio_cfg.get('mono', True),
    )
    return y.astype(np.float32), sr


def compute_log_mel(y: np.ndarray, sr: int, cfg: Dict) -> np.ndarray:
    audio_cfg = cfg['audio']
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=audio_cfg['n_fft'],
        hop_length=audio_cfg['hop_length'],
        win_length=audio_cfg['win_length'],
        n_mels=audio_cfg['n_mels'],
        fmin=audio_cfg['fmin'],
        fmax=audio_cfg['fmax'],
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=audio_cfg.get('top_db', 80))
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
    return mel_db.astype(np.float32)


def compute_rhythm_guides(y: np.ndarray, sr: int, cfg: Dict) -> Tuple[np.ndarray, np.ndarray, float]:
    hop = cfg['audio']['hop_length']
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, hop_length=hop, backtrack=False)
    tempo, beat_frames = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr, hop_length=hop)
    onset_times = librosa.frames_to_time(onset_frames, sr=sr, hop_length=hop) * 1000.0
    beat_times = librosa.frames_to_time(beat_frames, sr=sr, hop_length=hop) * 1000.0
    return onset_times.astype(np.float32), beat_times.astype(np.float32), float(np.asarray(tempo).reshape(-1)[0])


def augment_waveform(y: np.ndarray, sr: int, *, pitch_shift_steps: float = 0.0, time_stretch_rate: float = 1.0) -> np.ndarray:
    augmented = np.asarray(y, dtype=np.float32)
    if abs(float(pitch_shift_steps)) > 1e-8:
        augmented = librosa.effects.pitch_shift(augmented, sr=sr, n_steps=float(pitch_shift_steps))
    if abs(float(time_stretch_rate) - 1.0) > 1e-8:
        augmented = librosa.effects.time_stretch(augmented, rate=float(time_stretch_rate))
    return np.asarray(augmented, dtype=np.float32)


def extract_audio_features(audio_path: str | Path, cfg: Dict) -> AudioFeatures:
    y, sr = load_audio(audio_path, cfg)
    return extract_audio_features_from_waveform(y, sr, cfg)


def extract_audio_features_from_waveform(y: np.ndarray, sr: int, cfg: Dict) -> AudioFeatures:
    mel = compute_log_mel(y, sr, cfg)
    onset_ms, beat_ms, bpm = compute_rhythm_guides(y, sr, cfg)
    duration_ms = len(y) / sr * 1000.0
    return AudioFeatures(
        waveform=np.asarray(y, dtype=np.float32),
        sr=sr,
        duration_ms=duration_ms,
        mel=mel,
        onset_times_ms=onset_ms,
        beat_times_ms=beat_ms,
        bpm=bpm,
    )


def frame_times_ms(num_frames: int, cfg: Dict) -> np.ndarray:
    hop = cfg['audio']['hop_length']
    sr = cfg['audio']['sample_rate']
    return (np.arange(num_frames, dtype=np.float32) * hop / sr) * 1000.0



In [ ]:
%%writefile src/beatbert/utils/midi.py
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Dict, List, Tuple

import mido
import numpy as np


@dataclass
class MidiNote:
    pitch: int
    velocity: int
    start_ms: float
    end_ms: float
    lane: int


def _lane_from_pitch(pitch: int, cfg: Dict) -> int:
    midi_cfg = cfg['midi']
    strategy = midi_cfg['lane_strategy']
    num_lanes = int(midi_cfg['num_lanes'])
    if strategy == 'explicit':
        mapping = {int(k): int(v) for k, v in midi_cfg.get('explicit_pitch_map', {}).items()}
        if pitch not in mapping:
            raise KeyError(f'Pitch {pitch} not found in explicit_pitch_map')
        return mapping[pitch]
    if strategy == 'modulo':
        return int(pitch) % num_lanes
    if strategy == 'range':
        min_pitch = int(midi_cfg.get('min_pitch', 21))
        max_pitch = int(midi_cfg.get('max_pitch', 108))
        clipped = min(max(pitch, min_pitch), max_pitch)
        normalized = (clipped - min_pitch) / max(1, max_pitch - min_pitch + 1)
        return min(int(normalized * num_lanes), num_lanes - 1)
    raise ValueError(f'Unsupported lane strategy: {strategy}')


def parse_midi_notes(midi_path: str | Path, cfg: Dict) -> List[MidiNote]:
    midi = mido.MidiFile(str(midi_path))
    tempo = 500000
    current_ms = 0.0
    ticks_per_beat = midi.ticks_per_beat or cfg['midi'].get('ticks_per_beat_default', 480)
    active: Dict[Tuple[int, int], List[Tuple[float, int]]] = {}
    notes: List[MidiNote] = []
    velocity_threshold = int(cfg['midi'].get('velocity_threshold', 1))

    merged = mido.merge_tracks(midi.tracks)
    for msg in merged:
        delta_ms = mido.tick2second(msg.time, ticks_per_beat, tempo) * 1000.0
        current_ms += delta_ms
        if msg.type == 'set_tempo':
            tempo = msg.tempo
            continue
        if msg.type == 'note_on' and msg.velocity >= velocity_threshold:
            key = (msg.channel if hasattr(msg, 'channel') else 0, msg.note)
            active.setdefault(key, []).append((current_ms, msg.velocity))
            continue
        if msg.type in {'note_off', 'note_on'}:
            is_off = msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0)
            if not is_off:
                continue
            key = (msg.channel if hasattr(msg, 'channel') else 0, msg.note)
            if key not in active or len(active[key]) == 0:
                continue
            start_ms, velocity = active[key].pop(0)
            end_ms = max(current_ms, start_ms)
            lane = _lane_from_pitch(msg.note, cfg)
            notes.append(MidiNote(
                pitch=msg.note,
                velocity=velocity,
                start_ms=start_ms,
                end_ms=end_ms,
                lane=lane,
            ))

    notes.sort(key=lambda n: n.start_ms)
    return notes


def scale_note_timestamps(notes: List[MidiNote], factor: float) -> List[MidiNote]:
    if factor <= 0:
        raise ValueError('scale factor must be positive')
    if abs(factor - 1.0) < 1e-8:
        return list(notes)
    scaled: List[MidiNote] = []
    for note in notes:
        scaled.append(replace(
            note,
            start_ms=float(note.start_ms) * factor,
            end_ms=float(note.end_ms) * factor,
        ))
    return scaled


def notes_to_frame_labels(notes: List[MidiNote], frame_times_ms: np.ndarray, cfg: Dict) -> Dict[str, np.ndarray]:
    num_frames = len(frame_times_ms)
    event = np.zeros(num_frames, dtype=np.float32)
    lane = np.full(num_frames, fill_value=-100, dtype=np.int64)

    for note in notes:
        idx = int(np.argmin(np.abs(frame_times_ms - note.start_ms)))
        event[idx] = 1.0
        lane[idx] = int(note.lane)

    return {
        'event': event,
        'lane': lane,
    }



In [ ]:
%%writefile src/beatbert/utils/io.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict

import numpy as np


def save_npz(path: str | Path, **arrays: Any) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    np.savez(path, **arrays)  # TANPA kompresi — loading 5-10x lebih cepat saat training


def load_npz(path: str | Path) -> Dict[str, Any]:
    data = np.load(path, allow_pickle=True)
    return {k: data[k] for k in data.files}


def write_json(path: str | Path, data: Dict[str, Any]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)



In [ ]:
%%writefile src/beatbert/utils/seed.py
import os
import random

import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)



In [ ]:
%%writefile src/beatbert/inference/__init__.py



In [ ]:
%%writefile src/beatbert/inference/postprocess.py
from __future__ import annotations

from typing import Dict, List

import numpy as np


def _snap(value: float, candidates: np.ndarray, tolerance_ms: float) -> float:
    if candidates.size == 0:
        return value
    idx = int(np.argmin(np.abs(candidates - value)))
    best = float(candidates[idx])
    return best if abs(best - value) <= tolerance_ms else value


def postprocess_events(events: List[Dict], onset_times_ms: np.ndarray, beat_times_ms: np.ndarray, cfg: Dict) -> List[Dict]:
    pcfg = cfg['postprocess']
    out = []
    last_global = -1e9
    last_by_lane = {lane: -1e9 for lane in range(cfg['model']['num_lanes'])}
    for e in sorted(events, key=lambda x: x['time_ms']):
        time_ms = float(e['time_ms'])
        lane = int(e['lane'])
        if pcfg.get('snap_to_onsets', True):
            time_ms = _snap(time_ms, onset_times_ms, pcfg['onset_snap_tolerance_ms'])
        if pcfg.get('snap_to_beats', True):
            time_ms = _snap(time_ms, beat_times_ms, pcfg['beat_snap_tolerance_ms'])
        if time_ms - last_global < pcfg['min_gap_ms']:
            continue
        if time_ms - last_by_lane[lane] < pcfg['same_lane_min_gap_ms']:
            continue
        e['time_ms'] = int(round(time_ms))
        out.append(e)
        last_global = time_ms
        last_by_lane[lane] = time_ms

    density_limit = float(pcfg.get('max_density_notes_per_second', 8))
    if density_limit <= 0:
        return out
    filtered = []
    window = []
    for e in out:
        t = e['time_ms']
        window = [x for x in window if t - x <= 1000]
        if len(window) < density_limit:
            filtered.append(e)
            window.append(t)
    return filtered



In [ ]:
%%writefile src/beatbert/inference/predictor.py
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import numpy as np
import torch

from beatbert.models.beatmap_model import BeatmapModel
from beatbert.inference.postprocess import postprocess_events
from beatbert.utils.audio import extract_audio_features, frame_times_ms
from beatbert.utils.io import write_json


def load_checkpoint_model(checkpoint_path: str | Path, cfg: Dict, device: torch.device) -> BeatmapModel:
    ckpt = torch.load(checkpoint_path, map_location=device)
    model = BeatmapModel(cfg).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model


def _run_chunked_inference(model: BeatmapModel, mel: np.ndarray, cfg: Dict, device: torch.device):
    total_frames = mel.shape[1]
    chunk = int(cfg['inference']['chunk_frames'])
    overlap = int(cfg['inference']['chunk_overlap'])
    step = max(1, chunk - overlap)

    event_accum = np.zeros(total_frames, dtype=np.float32)
    count_accum = np.zeros(total_frames, dtype=np.float32)
    lane_accum = np.zeros((total_frames, cfg['model']['num_lanes']), dtype=np.float32)

    with torch.no_grad():
        for start in range(0, total_frames, step):
            end = min(total_frames, start + chunk)
            piece = mel[:, start:end]
            valid_len = piece.shape[1]
            if valid_len < chunk:
                piece = np.pad(piece, ((0, 0), (0, chunk - valid_len)))
                mask_np = np.concatenate([np.ones(valid_len, dtype=np.float32), np.zeros(chunk - valid_len, dtype=np.float32)])
            else:
                mask_np = np.ones(chunk, dtype=np.float32)

            x = torch.from_numpy(piece).unsqueeze(0).to(device)
            mask = torch.from_numpy(mask_np).unsqueeze(0).to(device)
            outputs = model(x, mask)
            event_prob = torch.sigmoid(outputs['event_logits']).squeeze(0).cpu().numpy()[:valid_len]
            lane_prob = torch.softmax(outputs['lane_logits'], dim=-1).squeeze(0).cpu().numpy()[:valid_len]

            event_accum[start:end] += event_prob
            count_accum[start:end] += 1
            lane_accum[start:end] += lane_prob
            if end == total_frames:
                break

    count_accum = np.clip(count_accum, 1e-6, None)
    event_prob = event_accum / count_accum
    lane_prob = lane_accum / count_accum[:, None]
    return event_prob, lane_prob


def predict_song(audio_path: str | Path, checkpoint_path: str | Path, cfg: Dict, output_json_path: str | Path | None = None) -> Dict:
    device = torch.device('cuda' if torch.cuda.is_available() and cfg['general']['device'] != 'cpu' else 'cpu')
    model = load_checkpoint_model(checkpoint_path, cfg, device)
    feats = extract_audio_features(audio_path, cfg)
    mel = feats.mel
    frame_ms = frame_times_ms(mel.shape[1], cfg)

    event_prob, lane_prob = _run_chunked_inference(model, mel, cfg, device)
    lane_pred = lane_prob.argmax(axis=-1)

    threshold = cfg['postprocess']['event_threshold']
    raw_events: List[Dict] = []
    idxs = np.where(event_prob >= threshold)[0]
    for idx in idxs:
        raw_events.append({
            'time_ms': float(frame_ms[idx]),
            'lane': int(lane_pred[idx]),
            'confidence': float(event_prob[idx]),
        })

    final_events = postprocess_events(raw_events, feats.onset_times_ms, feats.beat_times_ms, cfg)
    result = {
        'song': Path(audio_path).name,
        'bpm': feats.bpm,
        'offset_ms': 0,
        'num_events': len(final_events),
        'events': final_events,
        'notes': [{'time_ms': int(round(e['time_ms'])), 'lane': int(e['lane'])} for e in final_events],
    }
    if output_json_path is not None:
        write_json(output_json_path, result)
    return result



In [ ]:
%%writefile scripts/preprocess_dataset.py
from pathlib import Path
import argparse
import sys

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from beatbert.configs import load_config, resolve_paths
from beatbert.data.preprocess import preprocess_dataset
from beatbert.utils.seed import set_seed


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    args = parser.parse_args()
    cfg = resolve_paths(load_config(args.config), ROOT)
    set_seed(cfg['seed'])
    df = preprocess_dataset(cfg)
    print(df.head())
    print(f'Processed {len(df)} songs -> {cfg["paths"]["processed_dir"]}')


if __name__ == '__main__':
    main()



In [ ]:
%%writefile scripts/make_splits.py
from pathlib import Path
import argparse
import sys

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from beatbert.configs import load_config, resolve_paths
from beatbert.data.splits import make_splits
from beatbert.utils.seed import set_seed


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    args = parser.parse_args()
    cfg = resolve_paths(load_config(args.config), ROOT)
    set_seed(cfg['seed'])
    train_df, val_df, test_df = make_splits(cfg)
    print(f'train={len(train_df)} val={len(val_df)} test={len(test_df)}')


if __name__ == '__main__':
    main()



In [ ]:
%%writefile scripts/train.py
from pathlib import Path
import argparse
import sys

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from beatbert.configs import load_config, resolve_paths
from beatbert.training.trainer import train
from beatbert.utils.seed import set_seed


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    args = parser.parse_args()
    cfg = resolve_paths(load_config(args.config), ROOT)
    set_seed(cfg['seed'])
    train(cfg)


if __name__ == '__main__':
    main()



In [ ]:
%%writefile scripts/evaluate.py
from pathlib import Path
import argparse
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from beatbert.configs import load_config, resolve_paths
from beatbert.data.dataset import BeatmapDataset
from beatbert.models.beatmap_model import BeatmapModel
from beatbert.training.metrics import decode_events, event_level_metrics


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--checkpoint', required=True)
    args = parser.parse_args()
    cfg = resolve_paths(load_config(args.config), ROOT)
    device = torch.device('cuda' if torch.cuda.is_available() and cfg['general']['device'] != 'cpu' else 'cpu')
    ds = BeatmapDataset(cfg, 'test')
    loader = DataLoader(ds, batch_size=1, shuffle=False)
    model = BeatmapModel(cfg).to(device)
    ckpt = torch.load(args.checkpoint, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    metrics = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(batch['mel'], batch['attention_mask'])
            pred_events = decode_events(
                outputs['event_logits'].squeeze(0).cpu().numpy(),
                outputs['lane_logits'].squeeze(0).cpu().numpy(),
                batch['frame_times_ms'].squeeze(0).cpu().numpy(),
                cfg['eval']['event_threshold'],
            )
            true_idxs = torch.where(batch['event'].squeeze(0) > 0.5)[0].cpu().numpy()
            frame_times = batch['frame_times_ms'].squeeze(0).cpu().numpy()
            lanes = batch['lane'].squeeze(0).cpu().numpy()
            true_events = [(float(frame_times[i]), int(lanes[i])) for i in true_idxs if frame_times[i] >= 0]
            metrics.append(event_level_metrics(pred_events, true_events, cfg['eval']['tolerance_ms']))

    avg = {k: float(np.nanmean([m[k] for m in metrics])) for k in metrics[0]}
    for k, v in avg.items():
        print(f'{k}: {v:.4f}')


if __name__ == '__main__':
    main()



In [ ]:
%%writefile scripts/predict.py
from pathlib import Path
import argparse
import sys

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from beatbert.configs import load_config, resolve_paths
from beatbert.inference.predictor import predict_song


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--checkpoint', required=True)
    parser.add_argument('--audio', required=True)
    parser.add_argument('--output', required=True)
    args = parser.parse_args()
    cfg = resolve_paths(load_config(args.config), ROOT)
    result = predict_song(args.audio, args.checkpoint, cfg, args.output)
    print(f"Saved {result['num_events']} events to {args.output}")


if __name__ == '__main__':
    main()



## 5. Write Config


In [ ]:
%%writefile configs/default.yaml
seed: 42

general:
  device: auto
  num_workers: 8       # RTX Pro 4500 pod: 32 vCPU → 8 workers aman
  pin_memory: true

paths:
  raw_dir: data/raw
  processed_dir: data/processed
  splits_dir: data/splits
  checkpoint_dir: checkpoints
  prediction_dir: data/predictions

audio:
  sample_rate: 22050
  mono: true
  n_fft: 2048
  hop_length: 256
  win_length: 2048
  n_mels: 128
  fmin: 30
  fmax: 11025
  top_db: 80

dataset:
  context_frames: 256
  train_stride: 256
  eval_stride: 256
  split_ratio:
    train: 0.7
    val: 0.15
    test: 0.15
  negative_sample_ratio: 0.30   # BALANCED: 0.15 terlalu sedikit, 0.5 terlalu banyak → 0.30 cukup untuk belajar negatif tanpa suppress recall

midi:
  ticks_per_beat_default: 480
  velocity_threshold: 1
  lane_strategy: range
  num_lanes: 4
  min_pitch: 21
  max_pitch: 108
  note_type_from_duration: false
  hold_threshold_ms: 999999

model:
  input_mels: 128
  cnn_channels: [32, 64, 128]
  kernel_size: 3
  d_model: 256                  # NAIK dari 128 → kapasitas lebih besar untuk decision boundary yang presisi
  num_heads: 8                  # NAIK dari 4 → lebih banyak attention head (d_model/num_heads = 32)
  num_layers: 4                 # NAIK dari 2 → transformer lebih dalam untuk diskriminasi lebih tajam
  ff_dim: 512                   # NAIK sesuai d_model
  ff_mult: 4                    # NAIK dari 2 → feedforward lebih lebar
  dropout: 0.15                 # TURUN dari 0.2 → model lebih besar butuh dropout lebih sedikit
  max_seq_len: 2048
  num_lanes: 4
  predict_note_type: false

train:
  seed: 42
  batch_size: 64                # NAIK dari 32 → RTX Pro 4500 32GB VRAM, bisa batch 64 tanpa masalah
  num_workers: 8                # NAIK dari 2 → match general.num_workers (32 vCPU)
  epochs: 80
  lr: 0.0003                    # TURUN dari 0.0005 → konvergensi lebih halus untuk model besar
  weight_decay: 0.02            # NAIK dari 0.01 → regularisasi lebih kuat → kurangi overfitting ke false positive
  warmup_epochs: 5
  min_lr_ratio: 0.01
  event_loss_weight: 1.0
  lane_loss_weight: 0.5
  early_stopping_patience: 15
  validate_every: 0             # 0 = skip validasi (hemat waktu & biaya RunPod), best.pt disimpan di epoch terakhir
  monitor_metric: f1            # Tetap F1, target > 0.7
  device: auto
  amp: true
  event_positive_weight: auto
  grad_clip: 1.0
  label_smoothing: 0.05         # TURUN dari 0.1 → prediksi lebih tajam (less smoothing = more confident boundaries)
  use_focal_loss: true
  focal_alpha: 0.65             # BALANCED: 0.80 terlalu recall-heavy, 0.50 terlalu precision-heavy → 0.65 sweet spot
  focal_gamma: 2.0              # REVERT ke 2.0 → gamma 3.0 terlalu agresif, membunuh recall

eval:
  event_threshold: 0.42         # BALANCED: 0.35 terlalu rendah (P=0.42), 0.50 terlalu tinggi (R=0.54) → 0.42 sweet spot

postprocess:
  event_threshold: 0.42         # Match eval threshold
  min_gap_ms: 95                # Sedikit lebih ketat dari 90, tapi tidak se-ketat 100
  same_lane_min_gap_ms: 130     # Antara 120 (lama) dan 140 (terlalu ketat)
  snap_to_onsets: true
  onset_snap_tolerance_ms: 30
  snap_to_beats: true
  beat_snap_tolerance_ms: 40
  max_density_notes_per_second: 7   # Antara 8 (lama) dan 6 (terlalu ketat)

# Augmentasi telah dihapus — 1000 lagu asli sudah cukup untuk training.

inference:
  chunk_frames: 512
  chunk_overlap: 128



## 6. Verify Existing Data
Data sudah di-download & di-preprocess sebelumnya. Cell ini hanya verifikasi.


In [ ]:
import os, numpy as np, pandas as pd

npz_files = [f for f in os.listdir('data/processed') if f.endswith('.npz')]
total_size = sum(os.path.getsize(os.path.join('data/processed', f)) for f in npz_files)
print(f"📦 NPZ files: {len(npz_files)} ({total_size / 1e9:.2f} GB)")

for split in ['train', 'val', 'test']:
    p = f'data/splits/{split}.csv'
    if os.path.exists(p):
        print(f"  ✓ {split}: {len(pd.read_csv(p))} songs")
    else:
        print(f"  ✗ {split}.csv NOT FOUND — run splits cell!")

if npz_files:
    with np.load(f'data/processed/{npz_files[0]}') as d:
        print(f"\n🔍 Sample: {npz_files[0]}")
        for k in d.files: print(f"  {k}: {d[k].shape} {d[k].dtype}")
    print("\n✅ Data ready for training!")


## 7. Training 🚀
20 epoch, **no validation** — best.pt disimpan di epoch terakhir.


In [ ]:
import os
for f in ['checkpoints/last.pt', 'checkpoints/best.pt']:
    if os.path.exists(f): os.remove(f); print(f"Removed: {f}")
print("✓ Fresh start.")


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from beatbert.configs import load_config, resolve_paths
from beatbert.training.trainer import train
from beatbert.utils.seed import set_seed
cfg = resolve_paths(load_config('configs/default.yaml'), os.getcwd())
set_seed(cfg['seed'])
print("=" * 60)
print(f"  Model: d={cfg['model']['d_model']}, L={cfg['model']['num_layers']}, H={cfg['model']['num_heads']}")
print(f"  Focal: alpha={cfg['train']['focal_alpha']}, gamma={cfg['train']['focal_gamma']}")
print(f"  Batch={cfg['train']['batch_size']}, LR={cfg['train']['lr']}, Epochs={cfg['train']['epochs']}")
print(f"  Validate every: {cfg['train'].get('validate_every', 1)} (0=skip)")
print("=" * 60)
train(cfg)
print("\n✅ Training complete!")


## 8. Post-Training Evaluation
Evaluasi **sekali** setelah 20 epoch.


In [ ]:
import sys, os, torch
from torch.utils.data import DataLoader
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from beatbert.configs import load_config, resolve_paths
from beatbert.data.dataset import BeatmapDataset
from beatbert.models.beatmap_model import BeatmapModel
from beatbert.training.trainer import validate
cfg = resolve_paths(load_config('configs/default.yaml'), os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load('checkpoints/best.pt', map_location=device)
model = BeatmapModel(cfg).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best.pt (epoch {ckpt.get('epoch','?')})")
val_ds = BeatmapDataset(cfg, 'val')
val_loader = DataLoader(val_ds, batch_size=cfg['train']['batch_size'], shuffle=False, num_workers=4, pin_memory=True)
val_loss, m = validate(model, val_loader, device, cfg, torch.tensor(1.0, device=device))
print(f"\nResults: loss={val_loss:.4f}")
for k, v in m.items(): print(f"  {k}: {v:.4f}")
print(f"\n{'✅' if m.get('f1',0)>0.7 else '⚠️'} F1={m.get('f1',0):.4f}")
print(f"{'✅' if m.get('precision',0)>0.7 else '⚠️'} P={m.get('precision',0):.4f}")
print(f"{'✅' if m.get('recall',0)>0.7 else '⚠️'} R={m.get('recall',0):.4f}")


## 9. Threshold Sweep


In [ ]:
import torch, numpy as np
model.eval()
all_l, all_t = [], []
with torch.no_grad():
    for b in val_loader:
        b = {k: v.to(device) for k, v in b.items()}
        o = model(b['mel'], b['attention_mask'])
        all_l.append(o['event_logits'].cpu())
        all_t.append(b['event'].cpu())
probs = torch.sigmoid(torch.cat(all_l))
tgts = torch.cat(all_t)
print(f"{'Thresh':>8} {'Prec':>8} {'Recall':>8} {'F1':>8}")
print("-" * 36)
bf, bt = 0, 0.5
for t in np.arange(0.30, 0.75, 0.05):
    p_ = (probs >= t).long(); t_ = tgts.long()
    tp=int(((p_==1)&(t_==1)).sum()); fp=int(((p_==1)&(t_==0)).sum()); fn=int(((p_==0)&(t_==1)).sum())
    p=tp/max(1,tp+fp); r=tp/max(1,tp+fn); f=2*p*r/max(1e-8,p+r)
    mk = " ←" if f > bf else ""
    if f > bf: bf, bt = f, t
    print(f"{t:>8.2f} {p:>8.4f} {r:>8.4f} {f:>8.4f}{mk}")
print(f"\n🎯 Best: threshold={bt:.2f}, F1={bf:.4f}")


## 10. Download Checkpoints


In [ ]:
import os, shutil
for f in ['checkpoints/best.pt', 'checkpoints/last.pt']:
    if os.path.exists(f):
        sz = os.path.getsize(f)/(1024*1024)
        shutil.copy2(f, f'/workspace/{os.path.basename(f)}')
        print(f"✓ {f} ({sz:.1f} MB) → /workspace/{os.path.basename(f)}")
print("\n💡 Download dari RunPod file manager: /workspace/best.pt")
